# Linly-Dubbing Colab WebUI
This notebook is optimized for running **Linly-Dubbing** in Google Colab with T4 GPU.

### 🛠️ Execution Guide
1.  **Step 1**: Initialize Conda environment (Kernel will restart).
2.  **Step 2**: Clone repository and submodules.
3.  **Step 3**: Install system and Python dependencies.
4.  **Step 4**: Apply runtime patches and download models.
5.  **Step 5**: Launch the WebUI.

In [26]:
# [Step 1] 初始化 Conda 环境 (Initialize Conda)
# 注意：执行此单元格后内核会自动重启 (Notebook will restart after execution)
try:
    import condacolab
    condacolab.check()
    print("Conda environment already initialized.")
except ImportError:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

✨🍰✨ Everything looks OK!
Conda environment already initialized.


In [27]:
# [Step 2] 获取代码 (Get Code)
import os
if not os.path.exists('/content/Linly-Dubbing'):
    %cd /content/
    !git clone https://github.com/infinite-gaming-studio/Linly-Dubbing.git --depth 1
else:
    print('Project already cloned. Pulling latest changes...')
    %cd /content/Linly-Dubbing
    !git pull

%cd /content/Linly-Dubbing
!git submodule update --init --recursive

Project already cloned. Pulling latest changes...
/content/Linly-Dubbing
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 7.26 KiB | 2.42 MiB/s, done.
From https://github.com/infinite-gaming-studio/Linly-Dubbing
   a151e6b..0b8d4ab  main       -> origin/main
Updating a151e6b..0b8d4ab
Fast-forward
 colab_webui.ipynb  | 624 ++++++++++++++++++++++++++++++++++++++++++++++++++---
 patch_colab_fix.py |  35 +--
 webui.py           |  31 ++-
 3 files changed, 634 insertions(+), 56 deletions(-)
/content/Linly-Dubbing


In [28]:
# [Step 3] 安装依赖 (Install Dependencies)
print("Installing system dependencies...")
!apt-get update -qq
!apt-get install -y -qq build-essential libfst-dev espeak-ng libsndfile1 \
    libavfilter-dev libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev > /dev/null

print("Installing ffmpeg and pynini via mamba...")
import condacolab
!mamba install -y ffmpeg==7.0.2 pynini==2.1.5 -c conda-forge > /dev/null

print("Installing Python requirements (this might take 2-3 minutes)...")
!pip install -q uv
!uv pip install --system --force-reinstall "numpy<2.0.0" "setuptools" "loguru" "yt-dlp" torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1

# Relax requirements for compatibility
!sed -i 's/numpy==1.26.3/numpy<2.0.0/g' requirements.txt

!uv pip install --system --force-reinstall -r requirements.txt
!uv pip install --system -r requirements_module.txt

print("✅ Dependencies installed.")

Installing system dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Installing ffmpeg and pynini via mamba...
Installing Python requirements (this might take 2-3 minutes)...
Using Python 3.11.11 environment at: /usr/local
Resolved 29 packages in 69ms                                         
Prepared 29 packages in 1ms                                              
Uninstalled 29 packages in 304ms
Installed 29 packages in 279ms                              
 ~ filelock==3.20.3
 - fsspec==2024.12.0
 + fsspec==2026.1.0
 ~ jinja2==3.1.6
 ~ loguru==0.7.3
 - markupsafe==2.1.5
 + markupsafe==3.0.3
 ~ mpmath==1.3.0
 - networkx==2.8.8
 + networkx==3.6.1
 ~ numpy==1.26.4
 ~ nvidia-cublas-cu12==12.1.3.1
 ~ nvidia-cuda-cupti-cu12==12.1.105
 ~ nvidia-cuda-nvrtc-cu12==12.1.105
 ~ nvidia-cuda-runtime-cu12==12.1.105
 ~ nvidia-cudnn-cu12==8.9.2.26
 ~ 

In [29]:
# [Step 4] 应用补丁与下载模型 (Apply Patches & Download Models)
# 运行补丁脚本解决 Matplotlib, TTS 和 Videotrans 的兼容性问题
!python patch_colab_fix.py

# 下载核心模型
print("Downloading models...")
!mkdir -p models/ASR/whisper
!wget -nc https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth \
    -O models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth
!python scripts/huggingface_download.py

Running all Colab patches...
Patching webui.py for Gradio compatibility...
webui.py already patched for Gradio.
Patching submodules/TTS/setup.py for Python 3.12 compatibility...
✅ TTS patched.
Patching Matplotlib backend for headless environment...
✅ Matplotlib backend set to 'Agg'.
⚠️ Could not find videotrans to patch. It might not be installed yet.
Verifying critical dependencies...
✅ yt_dlp is installed.
✅ loguru is installed.
✅ torch is installed.
✅ pynini is installed.
✅ All critical dependencies verified.
All patches completed.
File ‘models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth’ already there; not retrieving.
/usr/local/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/huggingface_hub/file_download.py:979:

In [30]:
# [Step 5] 启动 WebUI (Launch WebUI)
import os
os.environ['MPLBACKEND'] = 'Agg' # Headless Matplotlib

if not os.path.exists('.env'):
    !cp env.example .env

print("Starting WebUI... Please click the Public URL (gradio.live) once it appears.")
!python webui.py

Starting WebUI... Please click the Public URL (gradio.live) once it appears.
/usr/local/lib/python3.11/site-packages/pyannote/audio/core/io.py:43: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
/usr/local/lib/python3.11/site-packages/pyannote/audio/pipelines/speaker_verification.py:43: UserWarning: torchaudio._backend.get_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  backend = torchaudio.get_audio_backend()
/usr/local/lib/python3.11/site-packages/pyannote/audio/pipelines/speaker_verification.py:45: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained impor